# RoBERTa Final Model Training
This notebook trains a RoBERTa model for sequence classification using a predefined set of optimal hyperparameters. The dataset is split into training (80%), validation (10%), and test (10%) sets.

In [1]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report
)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-07-10 08:28:33.603694: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-10 08:28:34.443808: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
202

In [15]:
import wandb
import huggingface_hub

os.environ["WANDB_PROJECT"] = "roberta_degendered_final"

wandb.login(key="1ad06854b00e224ae562a79fe3c4cc218a95c08c")
# huggingface_hub.login(token="YOUR_HF_TOKEN")

wandb.init()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hice1/mwesley32/.netrc


In [ ]:
# Initialize W&B
wandb.init()

In [3]:
# Model and Hyperparameter Configuration
model_name = "roberta-base"
model_cache_path = "../scratch/cache/roberta_degendered_final"

hyperparameters = {
    "learning_rate": 2.91e-04,
    "num_train_epochs": 9,
    "per_device_train_batch_size": 32,
    "weight_decay": 0.02
}

In [4]:
# Data Preparation (80:10:10 Split)
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# First split: 80% train, 20% temp (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Second split: 10% validation, 10% test from the temp set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
# Tokenization Function
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding=False, max_length=512)
    tokens["labels"] = example["label"]
    return tokens

# Create Hugging Face Datasets
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_dataset = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_val = val_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

In [6]:
# Metrics Computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    
    # Get classification report
    report = classification_report(labels, preds, output_dict=True, zero_division=0, target_names=['Female', 'Male'])
    
    # Flatten the report for easy logging
    metrics = {
        'accuracy': report['accuracy'],
        'macro_avg_precision': report['macro avg']['precision'],
        'macro_avg_recall': report['macro avg']['recall'],
        'macro_avg_f1': report['macro avg']['f1-score'],
        'weighted_avg_precision': report['weighted avg']['precision'],
        'weighted_avg_recall': report['weighted avg']['recall'],
        'weighted_avg_f1': report['weighted avg']['f1-score'],
        'female_precision': report['Female']['precision'],
        'female_recall': report['Female']['recall'],
        'female_f1': report['Female']['f1-score'],
        'female_support': report['Female']['support'],
        'male_precision': report['Male']['precision'],
        'male_recall': report['Male']['recall'],
        'male_f1': report['Male']['f1-score'],
        'male_support': report['Male']['support']
    }
    
    print("Confusion Matrix:\n", confusion_matrix(labels, preds))

    return metrics

In [7]:
# Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Female", 1: "Male"},
    label2id={"Female": 0, "Male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Training Arguments
final_model_output_dir = "../scratch/final_roberta_degendered_model"
training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=hyperparameters["per_device_train_batch_size"],
    num_train_epochs=hyperparameters["num_train_epochs"],
    learning_rate=hyperparameters["learning_rate"],
    weight_decay=hyperparameters["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_avg_f1",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_roberta_degendered_training"
)

In [10]:
# Trainer Initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val, # Use validation set for in-training evaluation
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_605565/4000249224.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
# Train the Model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Avg Precision,Macro Avg Recall,Macro Avg F1,Weighted Avg Precision,Weighted Avg Recall,Weighted Avg F1,Female Precision,Female Recall,Female F1,Female Support,Male Precision,Male Recall,Male F1,Male Support
1,0.624400,0.629465,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
2,0.644500,0.618671,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
3,0.633900,0.618734,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
4,0.633100,0.622637,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
5,0.635600,0.620024,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
6,0.623600,0.619616,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
7,0.617200,0.618564,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
8,0.621300,0.618627,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
9,0.623800,0.618756,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000


Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]


TrainOutput(global_step=2025, training_loss=0.6236112604023498, metrics={'train_runtime': 172.9662, 'train_samples_per_second': 374.067, 'train_steps_per_second': 11.707, 'total_flos': 1.702354839284736e+16, 'train_loss': 0.6236112604023498, 'epoch': 9.0})

In [17]:
# Final Evaluation on the Test Set
print("--- Final Evaluation on Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")

print("\nFinal Test Set Evaluation Results:")
print(test_results)

--- Final Evaluation on Test Set ---


Confusion Matrix:
 [[  0 279]
 [  0 620]]

Final Test Set Evaluation Results:
{'test_loss': 0.6307249665260315, 'test_accuracy': 0.6896551724137931, 'test_macro_avg_precision': 0.3448275862068966, 'test_macro_avg_recall': 0.5, 'test_macro_avg_f1': 0.40816326530612246, 'test_weighted_avg_precision': 0.4756242568370987, 'test_weighted_avg_recall': 0.6896551724137931, 'test_weighted_avg_f1': 0.5629838142153414, 'test_female_precision': 0.0, 'test_female_recall': 0.0, 'test_female_f1': 0.0, 'test_female_support': 279.0, 'test_male_precision': 0.6896551724137931, 'test_male_recall': 1.0, 'test_male_f1': 0.8163265306122449, 'test_male_support': 620.0, 'test_runtime': 0.7423, 'test_samples_per_second': 1211.062, 'test_steps_per_second': 39.067, 'epoch': 9.0}


In [18]:
# Save the Final Model and Tokenizer
trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)
print(f"Final model and tokenizer saved to: {final_model_output_dir}")

Final model and tokenizer saved to: ../scratch/final_roberta_degendered_model
